This file is part of GaPSE
Copyright (C) 2022 Matteo Foglieni

GaPSE is free software: you can redistribute it and/or modify
it under the terms of the GNU General Public License as published by
the Free Software Foundation, either version 3 of the License, or
(at your option) any later version.

GaPSE is distributed in the hope that it will be useful, but
WITHOUT ANY WARRANTY; without even the implied warranty of
MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the GNU
General Public License for more details.

You should have received a copy of the GNU General Public License
along with GaPSE. If not, see <http://www.gnu.org/licenses/>.

# Iln_terms

Here we plot all the $I_\ell^n$ integrals that GaPSE uses to build every Two-Point
Correlation Function (TPCF), together with their asymptotic behaviour for
$s \rightarrow 0$.

All the plots and the data are saved in the `Iln_terms/` directory.

The formulas and the limits are only *stated* here; their derivation, with proof, is in
the [The $I_\ell^n$ integrals](../docs/src/IlnIntegrals.md) page of the manual.

## The theory in a nutshell

It is convenient to introduce the moments of the Power Spectrum

$$
    \sigma_i = \int_{k_\mathrm{min}}^{k_\mathrm{max}} \frac{\mathrm{d}q}{2 \pi^2} \,
        q^{2-i} \, P(q) \; ,
$$

which are the only property of $P(q)$ the limits depend on. Using the series expansion of
the spherical Bessel function near the origin,
$j_\ell(x) = x^\ell / (2\ell+1)!! + \mathcal{O}(x^{\ell+2})$, one gets

$$
    I_\ell^n(s) \; \underset{s \rightarrow 0}{\sim} \;
        \frac{\sigma_{n-\ell}}{(2\ell+1)!!} \, s^{\,\ell-n}
    \qquad \qquad
    \tilde{I}_0^4(s) \; \underset{s \rightarrow 0}{\sim} \; -\frac{\sigma_2}{6 \, s^2}
$$

so that the sign of $\ell - n$ decides everything:

| | $\ell$ | $n$ | $s \rightarrow 0$ | limit |
|:---|:---:|:---:|:---|:---|
| $I_0^0$ | 0 | 0 | $\sigma_0$ | const |
| $I_2^0$ | 2 | 0 | $\sigma_{-2} \, s^2 / 15$ | $0$ |
| $I_4^0$ | 4 | 0 | $\sigma_{-4} \, s^4 / 945$ | $0$ |
| $I_0^2$ | 0 | 2 | $\sigma_2 \, s^{-2}$ | $+\infty$ |
| $I_2^2$ | 2 | 2 | $\sigma_0 / 15$ | const |
| $I_3^1$ | 3 | 1 | $\sigma_{-2} \, s^2 / 105$ | $0$ |
| $I_1^3$ | 1 | 3 | $\sigma_2 \, s^{-2} / 3$ | $+\infty$ |
| $I_1^1$ | 1 | 1 | $\sigma_0 / 3$ | const |
| $\tilde{I}_0^4$ | - | - | $-\sigma_2 \, s^{-2} / 6$ | $+\infty$ |

The diverging ones are never a problem in practice: inside a TPCF they always come
multiplied by a $J$ carrying the matching positive power of $\Delta\chi$.

## GaPSE setup

In [ ]:
using Pkg
Pkg.activate(@__DIR__)

# This environment is (re)built here, so that a fresh clone - or a different
# Julia version - needs no manual setup step.
#  - GaPSE is NOT a registered package: an environment reaches it only through
#    the `path` entry that `Pkg.develop` writes into `Manifest.toml`. Checking
#    `Base.identify_package` alone is not enough: it resolves the UUID from
#    `Project.toml` and succeeds even when the source cannot be found.
#  - `Project.toml` holds names and UUIDs only, is Julia-version INdependent and
#    is tracked by git; it is rebuilt from `THEORY_DEPS` if it is missing.
#  - `Manifest.toml` holds the resolved versions, IS Julia-version specific and
#    is gitignored; `Pkg.resolve()` + `Pkg.instantiate()` rebuild it for the
#    Julia that is running. `resolve` first, or `instantiate` refuses whenever
#    `Project.toml` declares more than the manifest knows about.
# Nothing is downloaded when the environment is already complete.
let THEORY_DEPS = ["Plots", "PyPlot", "LaTeXStrings", "QuadGK",
                   "DelimitedFiles", "Printf", "SpecialFunctions"]
    declared = collect(keys(Pkg.project().dependencies))
    id = Base.identify_package("GaPSE")
    if !("GaPSE" in declared) || isnothing(id) || isnothing(Base.locate_package(id))
        @info "theory/: making the GaPSE of this repository available"
        Pkg.develop(path=dirname(@__DIR__))
    end
    todo = filter(d -> !(d in declared), THEORY_DEPS)
    isempty(todo) || (@info "theory/: adding the missing dependencies" todo; Pkg.add(todo))
    Pkg.resolve()
    Pkg.instantiate()
end

using GaPSE

using Plots, LaTeXStrings, QuadGK, DelimitedFiles, Printf
using SpecialFunctions: sphericalbesselj, gamma

# `pyplot()` needs a working matplotlib; `gr()` is used instead where it is absent
try
    pyplot()
catch err
    @warn "theory/: PyPlot cannot be initialised; falling back to the GR backend. " *
          "The figures will look slightly different." exception = err
    gr()
end

In [ ]:
const PATH_TO_GAPSE = normpath(joinpath(@__DIR__, ".."))

# Input matter Power Spectrum P(q) at z=0
const FILE_PS = joinpath(PATH_TO_GAPSE, "data", "WideA_ZA_pk.dat")

# Directory where the plots and the data will be saved
const DIR = joinpath(@__DIR__, "Iln_terms")
@assert isdir(DIR) "ERROR: DIR=$DIR DOESN'T EXIST!!!"

# Set this to `true` in order to save a copy of the plots where the
# documentation expects to find them.
const SAVE_TO_DOCS = true
const DOCS_ASSETS = joinpath(PATH_TO_GAPSE, "docs", "src", "assets", "Iln_terms")

# CAREFUL: these are NOT the integration extremes of the I_l^n!
# They are only the ones used by `IPSTools` for its own sigma_0, ..., sigma_4.
const K_MIN, K_MAX = 1e-6, 10.0

# THESE are the integration extremes of the I_l^n: `IPSTools` hard-codes
# `kmin, kmax, s0 = 1e-5, 1e3, 1e-3` (see `src/IPSTools.jl`) and hands them over to
# `xicalc`, no matter what `k_min` and `k_max` you pass to it.
# The sigma_i that appear in the asymptotic limits of the I_l^n must be computed with
# THESE extremes, otherwise the comparison is meaningless: sigma_2 barely notices the
# difference, but sigma_{-2} and sigma_{-4} change by 5 and 8 orders of magnitude.
const XICALC_KMIN, XICALC_KMAX = 1e-5, 1e3

# Comoving separations where the I_l^n will be evaluated
const SS = 10 .^ range(-6, 5, length=800)

# Comoving separations where the I_l^n will also be computed by direct quadrature.
# We stop at 1e-2 because there q*s <= 10 over the whole integration range, so the
# integrand is still perfectly resolved by `QUAD_GRID` below.
const SS_DIRECT = 10 .^ range(-6, -2, length=50)

# log-spaced grid used by `I_direct`
const QUAD_GRID = 10 .^ range(log10(XICALC_KMIN), log10(XICALC_KMAX), length=50_000)

In [ ]:
ips = GaPSE.InputPS(FILE_PS)
tools = GaPSE.IPSTools(ips; k_min=K_MIN, k_max=K_MAX, N=1024,
    fit_min=0.05, fit_max=0.5, con=true)

# P(q) evaluated once and for all on `QUAD_GRID`, so that `I_direct` is cheap
const PQ_GRID = [ips(q) for q in QUAD_GRID]

# The small-q power law of the MATTER Power Spectrum, P(q) = PS_AMP * q^N_P, as `InputPS`
# fits it on [1e-6, 3e-6] (with `con=false`, so that its `l_a` is exactly zero).
# It is the only property of P(q) the large-s behaviour depends on.
#
# CAREFUL: N_P is the small-k slope of the matter P(k), NOT the exponent `n_s - 1` of the
# dimensionless primordial curvature spectrum. The two are related by
# P_m(k) ~ k^4 T^2(k) P_R(k) ~ k^n_s T^2(k), with T -> 1 on large scales, so N_P = n_s.
# We measure it rather than assume it: for `WideA_ZA_pk.dat` it comes out +0.9600 .
const N_P, PS_AMP = ips.l_si, ips.l_b

In [ ]:
const SIGMA_CACHE = Dict{Int,Float64}()

"""
    sigma(i; kmin=XICALC_KMIN, kmax=XICALC_KMAX) ::Float64

Return the moment of the input Power Spectrum

```math
\\sigma_i = \\int_{k_\\mathrm{min}}^{k_\\mathrm{max}}
    \\frac{\\mathrm{d}q}{2 \\pi^2} \\, q^{2-i} \\, P(q) \\; .
```

`IPSTools` stores only ``\\sigma_0, ..., \\sigma_4``, while the asymptotic limits of
``I_2^0``, ``I_4^0`` and ``I_3^1`` need the negative-index ones, so we recompute them here.

The default extremes are the ones `IPSTools` uses for the ``I_\\ell^n`` themselves, and
NOT the `k_min`/`k_max` it uses for its own ``\\sigma_i``: with `WideA_ZA_pk.dat` the
difference is negligible for ``\\sigma_2`` (0.1%) but it is a factor ``5 \\times 10^4``
for ``\\sigma_{-2}`` and ``5 \\times 10^8`` for ``\\sigma_{-4}``, because those integrals
are completely dominated by their upper extreme.
"""
sigma(i; kmin=XICALC_KMIN, kmax=XICALC_KMAX) =
    get!(SIGMA_CACHE, i) do
        quadgk(q -> ips(q) * q^(2 - i) / (2 * π^2), kmin, 1e-1, 1e1, kmax)[1]
    end

"""
    dfact(n) ::Int

Return the double factorial ``n!!`` (with ``n!! = 1`` for ``n \\leq 0``).
"""
dfact(n) = n <= 0 ? 1 : prod(n:-2:1)

# name, l, n, the IntegralIPS stored in `tools`
const ILN = [
    ("I00", 0, 0, tools.I00),
    ("I20", 2, 0, tools.I20),
    ("I40", 4, 0, tools.I40),
    ("I02", 0, 2, tools.I02),
    ("I22", 2, 2, tools.I22),
    ("I31", 3, 1, tools.I31),
    ("I13", 1, 3, tools.I13),
    ("I11", 1, 1, tools.I11),
]

## Three things to keep in mind

In [ ]:
@printf("                     [%.0e, %.0e]        [%.0e, %.0e] \n",
    XICALC_KMIN, XICALC_KMAX, K_MIN, K_MAX)
for i in [-4, -2, 0, 2]
    @printf("sigma_%-3d = %14.6e \t %14.6e \n", i, sigma(i),
        quadgk(q -> ips(q) * q^(2 - i) / (2 * π^2), K_MIN, K_MAX)[1])
end

`IPSTools` stores only $\sigma_0, ..., \sigma_4$, while the asymptotes of $I_2^0$,
$I_4^0$ and $I_3^1$ need the negative-index ones, so we recompute them here — over the
extremes of point 2 above.

The direct quadrature. It is slow and usable only for $s \lesssim 10^{-2}$, where
the integrand still has no oscillation, but it is the only way to see the true
$s \rightarrow 0$ behaviour.

## The I_l^n by direct quadrature

In [ ]:
"""
    I_direct(l, n, s) ::Float64

Compute

```math
I_\\ell^n(s) = \\int_{k_\\mathrm{min}}^{k_\\mathrm{max}} \\frac{\\mathrm{d}q}{2\\pi^2}
    \\, q^2 \\, P(q) \\, \\frac{j_\\ell(qs)}{(qs)^n}
```

by brute-force trapezoidal quadrature on the log-spaced `QUAD_GRID`, with the same
extremes `xicalc` is given inside `IPSTools`.

This is slow and it is only usable for ``s \\lesssim 10^{-2}``, where the integrand still
has no oscillation, but it is the only way to see the true ``s \\rightarrow 0`` behaviour:
an `IntegralIPS` cannot show it, because below its `left` field it does not evaluate the
integral at all (see `plot_single`).
"""
function I_direct(l, n, s)
    integrand = [PQ_GRID[i] * QUAD_GRID[i]^3 * sphericalbesselj(l, QUAD_GRID[i] * s) /
                 (QUAD_GRID[i] * s)^n / (2 * π^2) for i in eachindex(QUAD_GRID)]
    lqs = log.(QUAD_GRID)
    return sum((integrand[i] + integrand[i+1]) * (lqs[i+1] - lqs[i]) / 2
               for i in 1:length(QUAD_GRID)-1)
end

"""
    I04_tilde_direct(s) ::Float64

Compute, by the same brute-force quadrature of `I_direct`,

```math
\\tilde{I}_0^4(s) = \\int_{k_\\mathrm{min}}^{k_\\mathrm{max}} \\frac{\\mathrm{d}q}{2\\pi^2}
    \\, q^2 \\, P(q) \\, \\frac{j_0(qs) - 1}{(qs)^4} \\; ,
```

i.e. the very same integral of `GaPSE.func_I04_tilde`.

Note that we CANNOT obtain it as `I_direct(0, 4, s) - sigma(4) / s^4`: the two terms are
equal up to ``\\mathcal{O}(s^2)``, so for small ``s`` the subtraction cancels every
significant digit. For the same reason the ratio ``(j_0(x)-1)/x^4`` is evaluated through
its series for ``x < 10^{-2}``, since `j_0(x) - 1` is pure round-off there.
"""
function I04_tilde_direct(s)
    # (j_0(x) - 1) / x^4 = -1/(6 x^2) + 1/120 - x^2/5040 + O(x^4)
    ratio(x) = x < 1e-2 ? -1 / (6 * x^2) + 1 / 120 - x^2 / 5040 :
               (sphericalbesselj(0, x) - 1) / x^4

    integrand = [PQ_GRID[i] * QUAD_GRID[i]^3 * ratio(QUAD_GRID[i] * s) / (2 * π^2)
                 for i in eachindex(QUAD_GRID)]
    lqs = log.(QUAD_GRID)
    return sum((integrand[i] + integrand[i+1]) * (lqs[i+1] - lqs[i]) / 2
               for i in 1:length(QUAD_GRID)-1)
end

In [ ]:
"""
    asymptote(s, l, n) ::Float64

Return the leading small-``s`` behaviour of ``I_\\ell^n``:

```math
I_\\ell^n(s) \\; \\xrightarrow[s \\rightarrow 0]{} \\;
    \\frac{\\sigma_{n-\\ell}}{(2\\ell+1)!!} \\, s^{\\,\\ell-n} \\; .
```

It is reached only for ``s \\ll 1/k_\\mathrm{max} = 10^{-3}\\, h_0^{-1}\\mathrm{Mpc}``,
which is where every term of the ``j_\\ell`` series but the first becomes negligible.
"""
asymptote(s, l, n) = sigma(n - l) * s^(l - n) / dfact(2 * l + 1)

"""
    asymptote_tilde(s) ::Float64

Return the leading small-``s`` behaviour of ``\\tilde{I}_0^4``, i.e.
``-\\sigma_2 / (6 \\, s^2)``.
"""
asymptote_tilde(s) = -sigma(2) / (6 * s^2)

"""
    mellin(l, mu) ::Float64

Return the Mellin transform of the spherical Bessel function,

```math
\\int_0^{+\\infty} \\mathrm{d}x \\, x^{\\mu-1} \\, j_\\ell(x) =
    \\sqrt{\\frac{\\pi}{2}} \\; 2^{\\,\\mu-3/2} \\;
    \\frac{\\Gamma\\left(\\frac{\\ell+\\mu}{2}\\right)}
          {\\Gamma\\left(\\frac{\\ell-\\mu+3}{2}\\right)} \\; ,
```

which converges for ``-\\ell < \\mu < 2`` and is meant as its analytic continuation
outside that strip.
"""
mellin(l, mu) = √(π / 2) * 2.0^(mu - 3 / 2) * gamma((l + mu) / 2) / gamma((l - mu + 3) / 2)

"""
    asymptote_large(s, l, n) ::Float64

Return the leading large-``s`` behaviour of ``I_\\ell^n``:

```math
I_\\ell^n(s) \\; \\xrightarrow[s \\rightarrow +\\infty]{} \\;
    \\frac{A}{2\\pi^2} \\, \\mathcal{M}_\\ell(\\mu) \\, s^{-(3+n_P)} \\; ,
    \\qquad \\mu = 3 + n_P - n \\; ,
```

with ``P(q) \\rightarrow A \\, q^{n_P}`` for ``q \\rightarrow 0`` and
``\\mathcal{M}_\\ell`` the `mellin` transform above. ``n_P`` is the small-``k`` slope of the
MATTER Power Spectrum (``\\simeq +0.96``), not the ``n_s - 1`` of the dimensionless
primordial curvature one.

It is NOT obtained by replacing ``P`` with its small-``q`` power law inside the integral:
that interchange is illegitimate, and for ``\\mu \\geq 2`` it produces a divergent
integral. It follows from the residue at the rightmost pole of the Mellin-Parseval
representation; see the manual.

The exponent does not depend on ``\\ell`` nor on ``n``: the ``s^{-n}`` coming from the
``(qs)^{-n}`` factor exactly cancels the ``n`` carried by ``\\mu``.

It holds as long as the region ``q \\sim 1/s`` that dominates the integral still lies
inside the ``P \\propto q^{n_P}`` regime, i.e. for
``1 \\ll s \\ll 1/k_\\mathrm{min} = 10^5 \\, h_0^{-1}\\mathrm{Mpc}``.
"""
asymptote_large(s, l, n) = PS_AMP / (2 * π^2) * mellin(l, 3 + N_P - n) * s^(-(3 + N_P))

## Plot functions

In [ ]:
"""
    plot_kwargs(kwargs...) :: Dict

The defaults every figure of this directory shares, with anything in `kwargs...`
overriding them. `merge` keeps the value of the **last** dictionary for a repeated
key, so whatever is passed in always wins:

```julia
plot_kwargs()                 # the defaults
plot_kwargs(:dpi => 150)      # the defaults, with dpi = 150
```
"""
function plot_kwargs(kwargs...)
    dict_defaults = Dict(
        :size => (1000, 400), :dpi => 300, :legendposition => :outerright,
        :legendfontsize => 11, :guidefontsize => 14, :tickfontsize => 11,
        :titlefontsize => 16, :left_margin => 10Plots.mm, :bottom_margin => 6Plots.mm,
        # a y label that reads horizontally: see `hlabel` for the padding it needs
        :yguidefontrotation => -90,
    )
    # in `merge`, if a key is repeated the LAST collection has priority
    merge(dict_defaults, Dict(kwargs))
end

"""
    logticks(lo, hi; step=1) :: Vector{Float64}
    logticks(xs; step=1) :: Vector{Float64}

Decade ticks covering `[lo, hi]`, one every `step` decades. The second form reads the
range off the extrema of `xs`.
"""
logticks(lo, hi; step=1) = 10.0 .^ (floor(Int, log10(lo)):step:ceil(Int, log10(hi)))
logticks(xs; step=1) = logticks(extrema(xs)...; step=step)

"""
    logticks_minor(lo, hi) :: Tuple{Vector,Vector}

Decade ticks with the nine minor ticks of each decade, the major ones labelled and the
minor ones not: `(positions, labels)`, ready for `xticks = ...`.

Generating them from the plotted range, instead of once and for all, is what keeps the
labels of the left edge from piling up on each other when a figure does not span all the
decades. Use `logticks` instead when the minor ticks are not wanted.
"""
function logticks_minor(lo, hi)
    b_min, b_max = floor(Int, log10(lo)), ceil(Int, log10(hi))
    ts = [a * 10.0^b for b in b_min:b_max for a in 1:9]
    ls = [a == 1 ? L"10^{%$b}" : nothing for b in b_min:b_max for a in 1:9]
    keep = lo .<= ts .<= hi
    return (ts[keep], ls[keep])
end

"""
    hlabel(s; pad=10) :: String

A y-axis label meant to be read horizontally, i.e. together with
`yguidefontrotation = -90` (which `plot_kwargs` sets by default).

`Plots` places the y guide at a fixed offset from the axis, measured as if the label
were vertical. Once it is rotated flat it lands on top of the tick labels, and no
portable option moves it: padding with trailing spaces is the only thing that works
across backends. Keeping that in one function means only `pad` is tuned per figure.
"""
hlabel(s; pad=10) = s * " "^pad

"""
    save_plot(p, name) :: Nothing

Write `p` into `Iln_terms/`, and into the documentation assets when `SAVE_TO_DOCS`
is set. Plotting and saving are kept apart, so that a figure can be looked at and
tweaked before being written out.
"""
function save_plot(p, name)
    savefig(p, joinpath(DIR, name))
    SAVE_TO_DOCS && isdir(DOCS_ASSETS) && savefig(p, joinpath(DOCS_ASSETS, name))
    nothing
end

In [ ]:
"""
    shade_extrapolations!(p, iln; color, alpha, ls, lw, ss) :: Plots.Plot

Grey out the two regions where the `iln::IntegralIPS` does NOT evaluate the integral.

An `IntegralIPS` is a spline only between its `left` (`= fit_min = 0.05` for all the
``I_\\ell^n``, `0.1` for ``\\tilde{I}_0^4``) and its `right` fields; outside them it
returns a power law ``a + b \\, s^{\\,s_i}`` whose coefficients are fitted on
``[\\mathrm{fit\\_min}, \\mathrm{fit\\_max}] = [0.05, 0.5]`` (left) and on the last 16
points of the `xicalc` grid (right).

That extrapolation has nothing to do with the true ``s \\rightarrow 0`` behaviour, and
it is the single reason why a naive plot of an `IntegralIPS` down to ``s = 10^{-4}``
looks nothing like the analytic asymptote.
"""
function shade_extrapolations!(p, iln; ss=SS, color=:gray, alpha=0.13, ls=:dot, lw=1)
    ss[begin] < iln.left &&
        vspan!(p, [ss[begin], iln.left]; color=color, alpha=alpha, label="")
    iln.right < ss[end] &&
        vspan!(p, [iln.right, ss[end]]; color=color, alpha=alpha, label="")
    vline!(p, [iln.left, iln.right]; color=color, ls=ls, lw=lw, label="")
    return p
end


"""
    plot_single(name, l, n, f; tilde=false, ...) :: Plots.Plot

``|I_\\ell^n(s)|`` in log-log scale. Each figure carries four things:

 1. the `IntegralIPS` stored in `IPSTools` (solid), i.e. what GaPSE actually uses;
 2. the direct quadrature `I_direct` for ``s \\leq 10^{-2}`` (dotted), i.e. the true
    value of the integral there;
 3. the analytic small-``s`` asymptote (dashed, black) and, unless `tilde`, the
    large-``s`` one (dash-dotted, dark red);
 4. two grey bands, marking where the `IntegralIPS` is a power-law extrapolation and
    not the integral.

The absolute value is plotted because all these integrals oscillate and change sign
at large ``s``, where a logarithmic vertical axis would not be defined.

Every label, limit and style is a keyword, and anything else goes to `plot_kwargs`,
so a variant needs no editing of the function: `plot_single(...; shade=false)`.
"""
function plot_single(name, l, n, f;
    tilde=false, ss=SS, ss_direct=SS_DIRECT,
    xscale=:log10, yscale=:log10,
    xlabel=L"s \quad [h_0^{-1}\mathrm{Mpc}]",
    ylabel=hlabel(tilde ? L"|\tilde{I}_0^4(s)|" : L"|I_{\ell}^{n}(s)|"; pad=2),
    title=tilde ? L"\tilde{I}_0^4" : L"I_{%$l}^{%$n}",
    shade=true, pad_lo=30, pad_hi=30, lw=2, lw_direct=4,
    left_margin=18Plots.mm, minor_xticks=true,
    kwargs...
)
    ys = [f(s) for s in ss]
    dir = tilde ? [I04_tilde_direct(s) for s in ss_direct] :
          [I_direct(l, n, s) for s in ss_direct]

    # The asymptotes are drawn only over the side of the plot they describe: they are
    # pure power laws, so over the whole 11 decades of `ss` they would span 25 of them
    # and squash everything else.
    ss_asy = ss[ss.<=1.0]
    asy = tilde ? [asymptote_tilde(s) for s in ss_asy] :
          [asymptote(s, l, n) for s in ss_asy]
    ss_big = ss[ss.>=10.0]
    big = tilde ? nothing : [asymptote_large(s, l, n) for s in ss_big]

    lab = tilde ? L"|\tilde{I}_0^4(s)| \;\; \mathrm{(IPSTools)}" :
          L"|I_{%$l}^{%$n}(s)| \;\; \mathrm{(IPSTools)}"
    asylab = tilde ? L"|-\sigma_2 / (6 s^2)|" :
             L"|\sigma_{%$(n-l)} \, s^{%$(l-n)} / %$(dfact(2 * l + 1))|"

    # the vertical range is set by the data alone, letting the asymptotes clip
    vals = filter(v -> isfinite(v) && v > 0, abs.(vcat(ys, dir)))

    p = plot(; xscale=xscale, yscale=yscale, xlabel=xlabel, ylabel=ylabel, title=title,
        xlims=(ss[begin], ss[end]), ylims=(minimum(vals) / pad_lo, maximum(vals) * pad_hi),
        xticks=minor_xticks ? logticks_minor(ss[begin], ss[end]) : logticks(ss),
        plot_kwargs(:left_margin => left_margin, kwargs...)...)

    plot!(p, ss_asy, abs.(asy); label=asylab, ls=:dash, lw=lw, color=:black)
    tilde || plot!(p, ss_big, abs.(big);
        label=L"|A \, \mathcal{M}_{%$l}(\mu) \, s^{-(3+n_P)} / 2\pi^2|",
        ls=:dashdot, lw=lw, color=:darkred)
    plot!(p, ss, abs.(ys); label=lab, lw=lw)
    plot!(p, ss_direct, abs.(dir); label=L"\mathrm{direct \; quadrature}",
        lw=lw_direct, ls=:dot)
    shade && shade_extrapolations!(p, f; ss=ss)
    return p
end


"""
    plot_all(; iln, ...) :: Plots.Plot

All the ``|I_\\ell^n(s)|`` together in a single log-log figure.

Only the region where the `IntegralIPS` are splines is shown, i.e.
``[\\mathrm{left}, \\mathrm{right}]``: outside it they are power-law extrapolations,
and plotting them there would be misleading.
"""
function plot_all(; iln=ILN, ss=SS,
    xscale=:log10, yscale=:log10,
    xlabel=L"s \quad [h_0^{-1}\mathrm{Mpc}]",
    ylabel=hlabel(L"|I_{\ell}^{n}(s)|"; pad=2),
    title=L"\mathrm{All \; the} \; I_{\ell}^{n}",
    show_tilde=true, lw=2, left_margin=18Plots.mm, minor_xticks=true,
    kwargs...
)
    left = maximum(f.left for (_, _, _, f) in iln)
    right = minimum(f.right for (_, _, _, f) in iln)
    sss = ss[left.<=ss.<=right]

    p = plot(; xscale=xscale, yscale=yscale, xlabel=xlabel, ylabel=ylabel, title=title,
        xlims=(left, right),
        xticks=minor_xticks ? logticks_minor(left, right) : logticks(left, right),
        plot_kwargs(:left_margin => left_margin, kwargs...)...)
    for (_, l, n, f) in iln
        plot!(p, sss, abs.([f(s) for s in sss]); label=L"I_{%$l}^{%$n}", lw=lw)
    end
    show_tilde && plot!(p, sss, abs.([tools.I04_tilde(s) for s in sss]);
        label=L"\tilde{I}_0^4", lw=lw, ls=:dot)
    return p
end


"""
    plot_ratios(; iln, ...) :: Plots.Plot

For each ``I_\\ell^n``, the ratio between the directly-computed integral and its
analytic small-``s`` asymptote.

Every curve must tend to 1 for ``s \\rightarrow 0``: this is the actual numerical
check of the limits derived in the manual. The convergence sets in only for
``s \\lesssim 1/k_\\mathrm{max} = 10^{-3} \\, h_0^{-1}\\mathrm{Mpc}``.
"""
function plot_ratios(; iln=ILN, ss=SS_DIRECT,
    xscale=:log10, yscale=:identity, ylims=(0, 1.3),
    xlabel=L"s \quad [h_0^{-1}\mathrm{Mpc}]",
    ylabel=hlabel(L"\frac{I_{\ell}^{n}(s)}{\mathrm{asymptote}(s)}"; pad=6),
    title=L"\mathrm{Convergence \; to \; the} \; s \rightarrow 0 \; \mathrm{limits}",
    lw=2, left_margin=30Plots.mm, minor_xticks=true,
    kwargs...
)
    p = plot(; xscale=xscale, yscale=yscale, ylims=ylims,
        xlabel=xlabel, ylabel=ylabel, title=title, xlims=(ss[begin], ss[end]),
        xticks=minor_xticks ? logticks_minor(ss[begin], ss[end]) : logticks(ss),
        plot_kwargs(:left_margin => left_margin, kwargs...)...)
    for (_, l, n, _) in iln
        plot!(p, ss, [I_direct(l, n, s) / asymptote(s, l, n) for s in ss];
            label=L"I_{%$l}^{%$n}", lw=lw)
    end
    hline!(p, [1.0]; color=:black, ls=:dash, lw=lw, label="")
    return p
end


"""
    plot_ratios_large_s(; iln, ...) :: Plots.Plot

For each ``I_\\ell^n``, the ratio between the `IntegralIPS` stored in `IPSTools` and
its analytic large-``s`` asymptote.

Every curve must tend to 1, and it does to better than 1% around
``s \\simeq 10^4 \\, h_0^{-1}\\mathrm{Mpc}``. Here there is no need for `I_direct`: the
whole range shown lies inside ``[\\mathrm{left}, \\mathrm{right}]``, so the
`IntegralIPS` IS the integral.

``\\tilde{I}_0^4`` is left out on purpose: its ``\\mu = n_P - 1 \\simeq -0.04`` sits on
the pole of ``\\Gamma(\\mu/2)``, which is precisely the ``\\sigma_4 / s^4`` divergence
its subtraction removes, so its two leading powers, ``s^{-(3+n_P)}`` and ``s^{-4}``,
are degenerate up to ``1 - n_P = 0.04`` and it never reaches a clean power law inside
its own validity window.
"""
function plot_ratios_large_s(; iln=ILN, ss=SS, lo=10.0,
    xscale=:log10, yscale=:identity, ylims=(0, 1.6),
    xlabel=L"s \quad [h_0^{-1}\mathrm{Mpc}]",
    ylabel=hlabel(L"\frac{I_{\ell}^{n}(s)}{\mathrm{asymptote}(s)}"; pad=6),
    title=L"\mathrm{Convergence \; to \; the} \; s \rightarrow +\infty \; \mathrm{limits}",
    lw=2, left_margin=30Plots.mm, minor_xticks=true,
    kwargs...
)
    hi = minimum(f.right for (_, _, _, f) in iln)
    sss = ss[lo.<=ss.<=hi]

    p = plot(; xscale=xscale, yscale=yscale, ylims=ylims,
        xlabel=xlabel, ylabel=ylabel, title=title, xlims=(lo, hi),
        xticks=minor_xticks ? logticks_minor(lo, hi) : logticks(lo, hi),
        plot_kwargs(:left_margin => left_margin, kwargs...)...)
    for (_, l, n, f) in iln
        plot!(p, sss, [f(s) / asymptote_large(s, l, n) for s in sss];
            label=L"I_{%$l}^{%$n}", lw=lw)
    end
    hline!(p, [1.0]; color=:black, ls=:dash, lw=lw, label="")
    return p
end

## Saving functions

In [ ]:
"""
    save_large_s_data()

Save in `Iln_terms/Iln_large_s_values.txt` the ``I_\\ell^n`` and their ratio to the
analytic large-``s`` asymptote, over the region where the `IntegralIPS` are splines.
"""
function save_large_s_data()
    lo, hi = 10.0, minimum(f.right for (_, _, _, f) in ILN)
    ss = SS[lo.<=SS.<=hi]

    out = joinpath(DIR, "Iln_large_s_values.txt")
    isfile(out) && rm(out)
    open(out, "w") do io
        println(io, GaPSE.BRAND)
        println(io, "#\n# The I_l^n and their ratio to the analytic large-s asymptote")
        println(io, "#   A / (2 pi^2) * M_l(mu) * s^-(3+n_P) ,   mu = 3 + n_P - n")
        println(io, "# with P(q) -> A q^n_P for q -> 0 . All the ratios must tend to 1 .")
        println(io, "#")
        println(io, "# From the `InputPS` left fit: n_P = $N_P , A = $PS_AMP")
        println(io, "# Shown only for $lo <= s <= $hi , where every I_l^n is a spline.")
        println(io, "#")
        println(io, "# s [h_0^{-1} Mpc] \t " *
                    join([n for (n, _, _, _) in ILN], " \t ") * " \t " *
                    join([n * "_ratio" for (n, _, _, _) in ILN], " \t "))
        for s in ss
            vals = [f(s) for (_, _, _, f) in ILN]
            rats = [v / asymptote_large(s, l, n) for (v, (_, l, n, _)) in zip(vals, ILN)]
            println(io, "$s \t " * join(vals, " \t ") * " \t " * join(rats, " \t "))
        end
    end
    return out
end

"""
    save_data()

Save in `Iln_terms/Iln_values.txt` a table with the comoving separations `s` and the
values of all the ``I_\\ell^n`` (and of ``\\tilde{I}_0^4``) there evaluated.

The header records both sets of ``\\sigma_i``: the ones over
``[k_\\mathrm{min}, k_\\mathrm{max}]`` that `IPSTools` stores, and the ones over
``[10^{-5}, 10^3]`` that enter the asymptotic limits.
"""
function save_data()
    out = joinpath(DIR, "Iln_values.txt")
    isfile(out) && rm(out)
    open(out, "w") do io
        println(io, GaPSE.BRAND)
        println(io, "#\n# The I_l^n integrals evaluated in the following comoving separations.")
        println(io, "# Input Power Spectrum file: $(basename(FILE_PS))")
        println(io, "#")
        println(io, "# The I_l^n are integrated over [$XICALC_KMIN, $XICALC_KMAX] (the extremes")
        println(io, "# `IPSTools` hands over to `xicalc`), and the sigma_i of the asymptotic")
        println(io, "# limits must use the same ones:")
        for i in [-4, -2, 0, 2, 4]
            println(io, "#   sigma_$i = $(sigma(i))")
        end
        println(io, "#")
        println(io, "# For comparison, the sigma_i stored by `IPSTools` (over [$K_MIN, $K_MAX]):")
        println(io, "#   sigma_0 = $(tools.σ_0) \t sigma_2 = $(tools.σ_2) \t sigma_4 = $(tools.σ_4)")
        println(io, "#")
        println(io, "# CAREFUL: outside [$(tools.I00.left), $(tools.I00.right)] " *
                    "(and [$(tools.I04_tilde.left), $(tools.I04_tilde.right)] for I04_tilde)")
        println(io, "# these values are power-law extrapolations, NOT the integrals.")
        println(io, "#")
        println(io, "# s [h_0^{-1} Mpc] \t " * join([n for (n, _, _, _) in ILN], " \t ") * " \t I04_tilde")
        for s in SS
            vals = [f(s) for (_, _, _, f) in ILN]
            println(io, "$s \t " * join(vals, " \t ") * " \t $(tools.I04_tilde(s))")
        end
    end
    return out
end

"""
    save_direct_data()

Save in `Iln_terms/Iln_direct_values.txt` the directly-computed ``I_\\ell^n(s)`` and the
ratio to their analytic asymptote, for the `SS_DIRECT` separations. Every ratio must tend
to 1 for ``s \\rightarrow 0``.
"""
function save_direct_data()
    out = joinpath(DIR, "Iln_direct_values.txt")
    isfile(out) && rm(out)
    open(out, "w") do io
        println(io, GaPSE.BRAND)
        println(io, "#\n# The I_l^n computed by direct quadrature (`I_direct`), and their ratio")
        println(io, "# to the analytic small-s asymptote sigma_{n-l} s^{l-n} / (2l+1)!! .")
        println(io, "# All the ratios must tend to 1 for s -> 0 .")
        println(io, "#")
        println(io, "# s [h_0^{-1} Mpc] \t " *
                    join([n for (n, _, _, _) in ILN], " \t ") * " \t " *
                    join([n * "_ratio" for (n, _, _, _) in ILN], " \t "))
        for s in SS_DIRECT
            vals = [I_direct(l, n, s) for (_, l, n, _) in ILN]
            rats = [v / asymptote(s, l, n) for (v, (_, l, n, _)) in zip(vals, ILN)]
            println(io, "$s \t " * join(vals, " \t ") * " \t " * join(rats, " \t "))
        end
    end
    return out
end

## One figure per integral

Each `I_l^n` against its two analytic asymptotes and the direct quadrature, with the
extrapolated regions shaded. Every label, limit and style is a keyword of
`plot_single`, and anything else is forwarded to `plot_kwargs`.

In [ ]:
ps = [plot_single(name, l, n, f) for (name, l, n, f) in ILN]
push!(ps, plot_single("I04_tilde", 0, 4, tools.I04_tilde; tilde=true))
ps[1]

## All of them together

Restricted to the region where the `IntegralIPS` really are splines.

In [ ]:
plot_all()

## The numerical check of the small-s limits

The ratio between the directly-computed integral and its asymptote. Every curve must
tend to 1, and does so only below `s ~ 1/k_max = 1e-3`.

In [ ]:
plot_ratios()

## The opposite end: the large-s limits

In [ ]:
plot_ratios_large_s()

## Save the plots and the data

In [ ]:
for (p, (name, _, _, _)) in zip(ps, ILN)
    save_plot(p, name * ".png")
end
save_plot(ps[end], "I04_tilde.png")
save_plot(plot_all(), "all_Iln.png")
save_plot(plot_ratios(), "ratios.png")
save_plot(plot_ratios_large_s(), "ratios_large_s.png")
println(save_data())

## END